# Hydrogeological and Geospatial Data for Surface and Subsurface Flow Modelling in Swedish Coastal Catchments Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant==1.0.22

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.8kx9-mx2j/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print("\nDataset ID:", metadata.id)
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets, fields and columns by their @id.
from mlcroissant._src.structure.graph.dataset import RecordSet

def get_record_sets(ds):
    # Fetch record sets from raw metadata (using @id for reference).
    # `ds.metadata` contains record sets under 'recordSet' or 'record_sets'
    # Because mlcroissant guarantees logical property names, we try both
    record_sets = getattr(ds.metadata, 'recordSets', None)
    if record_sets is None:
        record_sets = getattr(ds.metadata, 'recordSet', None)
    if not record_sets or not isinstance(record_sets, list):
        print("No 'recordSet' found in the dataset metadata.")
        return []
    return record_sets

record_sets = get_record_sets(dataset)

if record_sets:
    for idx, rec in enumerate(record_sets):
        print(f"[{idx+1}] RecordSet @id: {rec.id}")
        print(f"    Name: {getattr(rec, 'name', '<no name>')}")
        print(f"    Description: {getattr(rec, 'description', '<no description>')}")
        fields = getattr(rec, 'fields', [])
        if fields:
            print(f"    Fields (by @id):")
            for field in fields:
                print(f"      - {field.id}: {getattr(field, 'name', '')}")
        else:
            print("    (No fields listed)")
        print()
else:
    print("Dataset does not provide any record sets in the metadata via 'recordSet'.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify available record sets again by @id and load each to a DataFrame
record_set_ids = []
record_set_id_to_name = {}

rec_sets = getattr(dataset.metadata, 'recordSets', None) or getattr(dataset.metadata, 'recordSet', None)
if rec_sets:
    for r in rec_sets:
        record_set_ids.append(r.id)
        record_set_id_to_name[r.id] = getattr(r, 'name', r.id)
else:
    print("No record sets found.")

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)

# Show available dataframes' columns
if record_set_ids:
    sample_id = record_set_ids[0]
    print(f"Loaded DataFrame for RecordSet @id: {sample_id} ({record_set_id_to_name[sample_id]})")
    if not dataframes[sample_id].empty:
        print("Fields (column names):", dataframes[sample_id].columns.tolist())
        display(dataframes[sample_id].head())
    else:
        print("(DataFrame is empty)")
else:
    print("No dataframes available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA using a numeric field and a group field (columns must exist in the first record set)
#
# You may need to adjust the field IDs to a specific numeric field based on your data sets.

if record_set_ids:
    # Use the first dataframe as an example
    rec_id = record_set_ids[0]
    df = dataframes[rec_id]
    # Find a numeric field in the columns
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    # Fall back to manual selection if none found
    if not numeric_field:
        print("No numeric field found in columns:", df.columns.tolist())
    else:
        print(f"Using numeric field for EDA: {numeric_field}")
        # Filter: Keep rows with value above threshold (use 10 as example)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (total={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a possible categorical field (if present)
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            print(f"Grouping by categorical field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Plot histogram of the numeric field used in the EDA
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in Record Set: {rec_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped data is present, plot barplot
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df.head(20), x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field} (top 20)")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!--
The notebook demonstrated how to load, explore, and analyze the FAIR² Hydrogeological and Geospatial Data using the `mlcroissant` library, referencing all dataset structures by their `@id` fields, as required for robust, reproducible data science workflows.
-->